In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('bmw_cars_market_dataset_synthetic.csv')

In [11]:
df.size

190000

In [12]:
df.shape[0]

10000

In [13]:
df.shape[1]

19

In [14]:
df.dtypes

,0
model,object
year,int64
engine_size,float64
horsepower,int64
fuel_type,object
transmission,object
drivetrain,object
mileage_km,int64
fuel_consumption_l_per_100km,float64
co2_emissions_g_km,float64


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   car_id                        10000 non-null  int64  
 1   model                         10000 non-null  object 
 2   year                          10000 non-null  int64  
 3   engine_size                   9757 non-null   float64
 4   horsepower                    10000 non-null  int64  
 5   fuel_type                     10000 non-null  object 
 6   transmission                  9785 non-null   object 
 7   drivetrain                    10000 non-null  object 
 8   mileage_km                    10000 non-null  int64  
 9   fuel_consumption_l_per_100km  9754 non-null   float64
 10  co2_emissions_g_km            9775 non-null   float64
 11  price_usd                     10000 non-null  int64  
 12  doors                         10000 non-null  int64  
 13  se

In [26]:
if 'car_id' in df.columns:
    df.drop(columns=['car_id'], inplace=True)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model                         10000 non-null  object 
 1   year                          10000 non-null  int64  
 2   engine_size                   9757 non-null   float64
 3   horsepower                    10000 non-null  int64  
 4   fuel_type                     10000 non-null  object 
 5   transmission                  9785 non-null   object 
 6   drivetrain                    10000 non-null  object 
 7   mileage_km                    10000 non-null  int64  
 8   fuel_consumption_l_per_100km  9754 non-null   float64
 9   co2_emissions_g_km            9775 non-null   float64
 10  price_usd                     10000 non-null  int64  
 11  doors                         10000 non-null  int64  
 12  seats                         10000 non-null  int64  
 13  bo

In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('.', '_')
)


In [9]:
df.columns.tolist()

['model',
 'year',
 'engine_size',
 'horsepower',
 'fuel_type',
 'transmission',
 'drivetrain',
 'mileage_km',
 'fuel_consumption_l_per_100km',
 'co2_emissions_g_km',
 'price_usd',
 'doors',
 'seats',
 'body_type',
 'color',
 'owner_count',
 'accident_history',
 'service_history',
 'country_sold']

In [15]:
transmission_mode = df['transmission'].mode()[0]
df['transmission'] = df['transmission'].fillna(transmission_mode)

fuel_median = df['fuel_consumption_l_per_100km'].median()
df['fuel_consumption_l_per_100km'] = df['fuel_consumption_l_per_100km'].fillna(fuel_median)

df['service_history'] = df['service_history'].fillna('unknown')



In [16]:
int_cols = ['year', 'horsepower', 'mileage_km', 'co2_emissions_g_km', 'owner_count']
float_cols = ['engine_size', 'fuel_consumption_l_per_100km', 'price_usd']

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

In [17]:
cat_cols = [
    'model', 'fuel_type', 'transmission', 'drivetrain',
    'body_type', 'color', 'accident_history',
    'service_history', 'country_sold'
]

for col in cat_cols:
    df[col] = df[col].astype('category')


In [18]:
mapping = {
    'fuel_type': {'petrol': 'petrol', 'petrol ': 'petrol', 'diesel': 'diesel', 'electric': 'electric'},
    'transmission': {'auto': 'automatic', 'aut': 'automatic', 'man': 'manual'}
}

for col, replace_map in mapping.items():
    if col in df.columns:
        df[col] = df[col].str.lower().str.strip().replace(replace_map)



In [19]:
cols_to_fix = ['engine_size', 'horsepower', 'mileage_km', 'price_usd', 'fuel_consumption_l_per_100km']

for col in cols_to_fix:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)



### Feature Engineering

In [20]:
import datetime
current_year = 2025

df['car_age'] = current_year - df['year']

df['price_per_hp'] = df['price_usd'] / df['horsepower']

df['mileage_per_year'] = df['mileage_km'] / df['car_age'].replace(0, 1)

df['engine_efficiency'] = df['horsepower'] / df['engine_size']

In [22]:
df['accident_history_enc'] = df['accident_history'].map({'no': 0, 'yes': 1}).fillna(0)

service_map = {'full': 2, 'partial': 1, 'unknown': 0}
df['service_history_enc'] = df['service_history'].map(service_map).fillna(0)



In [24]:
def categorize_fuel(cons):
    if cons < 6: return 'efficient'
    elif 6 <= cons <= 9: return 'moderate'
    else: return 'inefficient'

df['fuel_efficiency_cat'] = df['fuel_consumption_l_per_100km'].apply(categorize_fuel)

luxury_list = ['5 Series', '7 Series', 'X5', 'X7', 'M5', 'M8']
df['luxury_model'] = df['model'].apply(lambda x: 1 if any(lux in str(x) for lux in luxury_list) else 0)



In [25]:
target_cols = [
    'price_usd',
    'mileage_km',
    'horsepower',
    'engine_size',
    'fuel_consumption_l_per_100km'
]

stats = df[target_cols].describe().T

summary = stats[['mean', '50%', 'min', 'max', 'std']]
summary.rename(columns={'50%': 'median'}, inplace=True)

print("\n--- Summary Statistics ---")
print(summary)


--- Summary Statistics ---
                                       mean   median      min         max  \
price_usd                      49636.435525  44324.0  2640.00  115593.375   
mileage_km                    112364.794000  94483.0     0.00  300000.000   
horsepower                       268.773800    260.0   120.00     476.000   
engine_size                        2.491022      2.6     0.05       5.000   
fuel_consumption_l_per_100km       6.691100      7.2     0.85      12.900   

                                       std  
price_usd                     27336.668638  
mileage_km                    86335.775989  
horsepower                       76.883380  
engine_size                       1.187666  
fuel_consumption_l_per_100km      2.755495  


/tmp/ipykernel_550/1847946882.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary.rename(columns={'50%': 'median'}, inplace=True)
